<a href="https://colab.research.google.com/github/Shitalsin/Python/blob/main/Assignment_Week_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
!pip install -q langchain
!pip install -q langchain-community
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q pypdf
!pip install -q transformers
!pip install -q accelerate
!pip install python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 5.7 MB/s eta 0:00:00


In [21]:
from google.colab import files
uploaded=files.upload()

Saving Reading Comprehension Practice Questions.docx to Reading Comprehension Practice Questions.docx


In [22]:
!pip install docx2txt

In [26]:
# Reading the documents/pdf
from langchain_community.document_loaders import Docx2txtLoader
loader = Docx2txtLoader("Reading Comprehension Practice Questions.docx")
documents = loader.load()
print(documents[0].page_content)

PASSAGE 1

Microfinance — the provision of small loans, savings accounts, and basic financial services to low-income individuals who lack access to conventional banking — emerged as a development tool in the 1970s. Its most celebrated pioneer, Muhammad Yunus, founded the Grameen Bank in Bangladesh in 1983 on the radical premise that the poor are creditworthy, and that access to small amounts of capital can enable them to build businesses, generate income, and escape poverty. The model spread rapidly, and by the early 2000s microfinance institutions operated in over a hundred countries, reaching tens of millions of borrowers.

The optimism surrounding microfinance, however, was not without its critics. Early randomised controlled trials conducted in India, Mexico, and Morocco found that while microcredit did help some borrowers smooth consumption and manage financial shocks, the transformative poverty-reduction effects that had been claimed were largely absent. Borrowers often used loan

In [28]:
import langchain
print(langchain.__version__)

1.3.13


In [29]:
!pip install -q langchain-text-splitters

In [33]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
chunks = text_splitter.split_documents(documents)
print("Total Chunks:", len(chunks))

Total Chunks: 25


In [37]:
print(chunks[1].page_content)

Microfinance — the provision of small loans, savings accounts, and basic financial services to low-income individuals who lack access to conventional banking — emerged as a development tool in the 1970s. Its most celebrated pioneer, Muhammad Yunus, founded the Grameen Bank in Bangladesh in 1983 on the radical premise that the poor are creditworthy, and that access to small amounts of capital can enable them to build businesses, generate income, and escape poverty. The model spread rapidly, and


In [39]:
from langchain_community.embeddings import HuggingFaceEmbeddings
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/tmp/ipykernel_1491/4211815356.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [42]:
# Create FAISS Vector Database(It is used to store vectors (embeddings) and quickly find the most similar vectors from millions or even billions of them.)
from langchain_community.vectorstores import FAISS
vector_db = FAISS.from_documents(
    chunks,
    embedding_model
)
print("Vector Database Created Successfully!")

Vector Database Created Successfully!


In [82]:
# Questing asking phase
question = input("Ask question?")

Ask question?Q3 The word 'predatory' as used in paragraph 3 most nearly means:


In [83]:
# retriving relevant chunks from the documents
retrieved_docs = vector_db.similarity_search(
    question,
    k=3
)
print("Retrieved Chunks:\n")

for i, doc in enumerate(retrieved_docs):
    print(f"\n----- Chunk {i+1} -----")
    print(doc.page_content)

Retrieved Chunks:


----- Chunk 1 -----
Q2 According to early research trials, what was a key finding about the impact of microcredit?	



A.  It eliminated poverty in all countries where it was introduced

B.  It helped borrowers manage financial shocks but did not produce transformative poverty reduction

C.  It caused widespread financial instability in developing countries

D.  It was most effective when interest rates exceeded thirty percent



Q3 The word 'predatory' as used in paragraph 3 most nearly means:

----- Chunk 2 -----
A more troubling development was the emergence of predatory lending practices in some markets. As microfinance attracted commercial investors seeking returns, institutions in countries such as Andhra Pradesh in India began competing aggressively for borrowers, issuing multiple loans to the same individuals and charging annualised interest rates that sometimes exceeded thirty percent. A wave of borrower over-indebtedness and, in some documented cases, suic

In [84]:
# Loading language model
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [85]:
# Create Prompt
context = "\n".join([doc.page_content for doc in retrieved_docs])

prompt = f"""
You are a helpful AI assistant.

Answer the question using ONLY the information provided in the context.

If the answer is not present in the context, reply:
"The answer is not available in the provided document."

Context:
{context}

Question:
{question}

Give a complete and concise answer.
"""



In [81]:
print(prompt)


You are a helpful AI assistant.

Answer the question using ONLY the information provided in the context.

If the answer is not present in the context, reply:
"The answer is not available in the provided document."

Context:
Q2 According to early research trials, what was a key finding about the impact of microcredit?	



A.  It eliminated poverty in all countries where it was introduced

B.  It helped borrowers manage financial shocks but did not produce transformative poverty reduction

C.  It caused widespread financial instability in developing countries

D.  It was most effective when interest rates exceeded thirty percent



Q3 The word 'predatory' as used in paragraph 3 most nearly means:
A more troubling development was the emergence of predatory lending practices in some markets. As microfinance attracted commercial investors seeking returns, institutions in countries such as Andhra Pradesh in India began competing aggressively for borrowers, issuing multiple loans to the same

In [88]:
# Generating Final answrer
inputs = tokenizer(prompt, return_tensors="pt", truncation=True)
outputs = model.generate(
    **inputs,
    max_new_tokens=100
)

answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Answer:\n")
print(answer)

Answer:

A more troubling development was the emergence of predatory lending practices in some markets.
